In [3]:
import sqlite3
import re


In [38]:
import pandas as pd

df_trucks = pd.read_excel(r"C:\Users\pstanam.MRX\Downloads\OneDrive_2026-02-23\McCraken Ticket System\Data to be imported for ticket system.xlsx",sheet_name="Trucks")
df_trucks.rename(columns={"Truck Number": "truck_number", "Driver Name": "driver_name","Trucking Company": "trucking_company", "License Plate": "license_plate"}, inplace=True)
df_trucks = df_trucks.iloc[1:].reset_index(drop=True)


df_customers = pd.read_excel(r"C:\Users\pstanam.MRX\Downloads\OneDrive_2026-02-23\McCraken Ticket System\Data to be imported for ticket system.xlsx",sheet_name="Customers")
df_customers['zip_code_str'] = df_customers['Zip Code'].apply(lambda x: str(int(x)) if pd.notnull(x) else '')
parts = ["Address Line 1","Address Line 2","City","State","zip_code_str",]
df_customers[parts] = df_customers[parts].fillna("").astype(str)
df_customers["full_Address"] = (df_customers[parts].agg(", ".join, axis=1).str.replace(r",\s*,", ",", regex=True).str.strip(", "))


df_materials = pd.read_excel(r"C:\Users\pstanam.MRX\Downloads\OneDrive_2026-02-23\McCraken Ticket System\Data to be imported for ticket system.xlsx",sheet_name="Material")
df_materials.rename(columns={'Rev# 11\n1 Axle':'Axle 1', 'Rev# 11\nTandem':'Tandem', 'Rev# 11\nTriAxle':'TriAxle', 'Rev# 11\n4/5 Axle':'Axle 4_5', 'Rev# 11\n6 Axle':'Axle 6', 'Rev# 11\nSemi':'Semi', 'Rev# 11\nHydVac':'HydVac','Rev# 11\nDIRT IN':'DIRT IN'}, inplace=True)
pattern = r"\s*\((IN|OUT)\)\s*$"

# 1) On df_materials
df_materials["direction_from_material"] = (
    df_materials["Material"].fillna("")
    .astype("str")
    .str.extract(pattern, expand=False ,flags=re.IGNORECASE)
    .str.upper()
)

df_materials["material_clean"] = (
    df_materials["Material"].fillna("")
    .astype("str")
    .str.replace(pattern, "", regex=True, flags=re.IGNORECASE)
    .str.strip()
)
df_materials['direction_from_material'] = df_materials['direction_from_material'].fillna("IN").str.upper()
df_materials.drop(columns=["Material",'Direction','Material new name'], inplace=True)

In [39]:
df_materials

,Cat,Code,Axle 1,Tandem,TriAxle,Axle 4_5,Axle 6,Semi,HydVac,DIRT IN,direction_from_material,material_clean
0,41,RCC12,140.0,168.0,224.0,308.0,336.0,364.0,NaN,NaN,OUT,#1&2 Recycled Concrete
1,33,10,350.0,420.0,560.0,770.0,840.0,910.0,NaN,NaN,OUT,#10 Limestone
2,32,304,350.0,420.0,560.0,770.0,840.0,910.0,NaN,NaN,OUT,#304 Limestone
3,40,RCC304,125.0,150.0,200.0,275.0,300.0,325.0,NaN,NaN,OUT,#304 Recycled Concrete
4,34,57,400.0,480.0,640.0,880.0,960.0,1040.0,NaN,NaN,OUT,#57 Limestone
5,42,RCC57,150.0,180.0,240.0,330.0,360.0,390.0,NaN,NaN,OUT,#57 Recycled Concrete
6,1,Grin,20.0,24.0,32.0,44.0,48.0,52.0,NaN,NaN,IN,Asphalt Grindings (no chunks)
7,22,Ogrin,80.0,96.0,128.0,176.0,192.0,208.0,NaN,NaN,OUT,Asphalt Grindings
8,12,Deck,625.0,750.0,1000.0,1375.0,1500.0,1625.0,NaN,NaN,IN,Bridge Deck
9,8,Asph,125.0,150.0,200.0,275.0,300.0,325.0,NaN,NaN,IN,Broken Asphalt


,truck_number,trucking_company,Notes,Size,Phone,license_plate
0,LAWN01,44 LAWNCARE LANDSCAPING SERVICES,NaN,1.0,NaN,PLM7176
1,ACE74,ACE Trucking,AL,5.0,NaN,NaN
2,ACE75,ACE Trucking,NaN,5.0,NaN,NaN
3,ACE76,ACE Trucking,NaN,5.0,NaN,PLB7535
4,BEP13,BEP TRUCKING,NaN,7.0,330-483-4252,TRF5008


In [40]:
import psycopg2

conn = psycopg2.connect(
    user="tktsys_pgdb",
    password="Mccraken_database",
    host="tktsys-db.postgres.database.azure.com",
    port=5432,
    database="postgres"
)

cursor = conn.cursor()

cursor.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public';
""")

tables = cursor.fetchall()

for t in tables:
    print(t[0])

cursor.close()
conn.close()

material_price
customers
trucks_main
trucks
materials
ticket_sequence
jobs_cache
tickets
manual_jobs


In [6]:
import psycopg2

conn = psycopg2.connect(
    host="tktsys-db.postgres.database.azure.com",
    database="postgres",
    user="tktsys_pgdb",
    password="Mccraken_database",
    port=5432,
    sslmode="require"
)

cur = conn.cursor()
cur.execute("""
CREATE TABLE IF NOT EXISTS material_price (
    id SERIAL PRIMARY KEY,
    cat INTEGER NOT NULL,
    material TEXT NOT NULL UNIQUE,
    direction TEXT NOT NULL CHECK (direction IN ('IN','OUT')),
    axle1 DOUBLE PRECISION,
    axle2 DOUBLE PRECISION,
    axle3 DOUBLE PRECISION,
    axle4 DOUBLE PRECISION,
    axle5 DOUBLE PRECISION,
    axle6 DOUBLE PRECISION,
    axle7 DOUBLE PRECISION,
    axle8 DOUBLE PRECISION,
    axle9 DOUBLE PRECISION,
    active BOOLEAN NOT NULL DEFAULT TRUE
);
""")

conn.commit()
print("Table material_price created successfully!")
cur.close()
conn.close()

Table material_price created successfully!


In [41]:
conn = psycopg2.connect(
    host="tktsys-db.postgres.database.azure.com",
    database="postgres",
    user="tktsys_pgdb",
    password="Mccraken_database",
    port=5432,
    sslmode="require"
)

cur = conn.cursor()
cur.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema='public'
""")

print(cur.fetchall())

[('material_price',), ('customers',), ('trucks_main',), ('trucks',), ('materials',), ('ticket_sequence',), ('jobs_cache',), ('tickets',), ('manual_jobs',)]


In [4]:
conn = psycopg2.connect(
    host="tktsys-db.postgres.database.azure.com",
    database="postgres",
    user="tktsys_pgdb",
    password="Mccraken_database",
    port=5432,
    sslmode="require"
)

cur = conn.cursor()
cur.execute("""
SELECT *
FROM jobs_cache
LIMIT 5
""")

print(cur.fetchall())

[(3609, '2723', 'CSCC Health Sciences (2723)', 'Redcon, LLC', 0, None, '2026-03-18T13:55:52'), (3610, '2733', 'Foxconn Lordstown (2733)', 'Redcon, LLC', 0, None, '2026-03-18T13:55:52'), (3611, '2734', 'Earlington Park (2734)', 'Redcon, LLC', 0, None, '2026-03-18T13:55:52'), (3612, '2739', 'Spitzer Hyundai-Cleve (2739)', 'Mr. Excavator, Inc.', 0, None, '2026-03-18T13:55:52'), (3613, '2741', 'Mentor Pool (2741)', 'Eagle Abatement & Demolition', 1, None, '2026-03-18T13:55:52')]


In [43]:
cur = conn.cursor()
cur.execute("""
SELECT *
FROM material_price
LIMIT 5
""")

for i in cur.fetchall():
    print(i)

(1, 41, '#1&2 Recycled Concrete', 'OUT', 140.0, 168.0, 224.0, 308.0, 308.0, 336.0, 364.0, nan, nan, True)
(2, 33, '#10 Limestone', 'OUT', 350.0, 420.0, 560.0, 770.0, 770.0, 840.0, 910.0, nan, nan, True)
(3, 32, '#304 Limestone', 'OUT', 350.0, 420.0, 560.0, 770.0, 770.0, 840.0, 910.0, nan, nan, True)
(4, 40, '#304 Recycled Concrete', 'OUT', 125.0, 150.0, 200.0, 275.0, 275.0, 300.0, 325.0, nan, nan, True)
(5, 34, '#57 Limestone', 'OUT', 400.0, 480.0, 640.0, 880.0, 880.0, 960.0, 1040.0, nan, nan, True)


In [25]:
conn = psycopg2.connect(
    host="tktsys-db.postgres.database.azure.com",
    database="postgres",
    user="tktsys_pgdb",
    password="Mccraken_database",
    port=5432,
    sslmode="require"
)

cur = conn.cursor()
for index, row in df_materials.iterrows():
    cur.execute("""
        INSERT INTO material_price 
        (cat, material, direction, axle1, axle2, axle3, axle4, axle5, axle6, axle7, axle8, axle9)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        row['Cat'],
        row['Material new name'],
        row['Direction'],
        row['1'],
        row['2'],
        row['3'],
        row['4'],
        row['5'],
        row['6'],
        row['7'],
        row['8'],
        row['9']
    ))

conn.commit()

print("Data inserted successfully into material_price!")
cur.close()
conn.close()

Data inserted successfully into material_price!


In [26]:
conn = psycopg2.connect(
    host="tktsys-db.postgres.database.azure.com",
    database="postgres",
    user="tktsys_pgdb",
    password="Mccraken_database",
    port=5432,
    sslmode="require"
)

cur = conn.cursor()
cur.execute("""
SELECT id
FROM material_price
""")

print(cur.fetchall())

[(1,), (2,), (3,), (4,), (5,), (6,), (7,), (8,), (9,), (10,), (11,), (12,), (13,), (14,), (15,), (16,), (17,), (18,), (19,), (20,), (21,), (22,), (23,), (24,), (25,), (26,), (27,)]


In [27]:
cur.execute("""
CREATE TABLE IF NOT EXISTS customers (
    id SERIAL PRIMARY KEY,
    customer_name TEXT,
    full_address TEXT,
    contact_person TEXT,
    phone_number TEXT,
    notes TEXT
);
""")

conn.commit()
print("Table customers created successfully!")

Table customers created successfully!


In [28]:
cur.execute(
    """CREATE TABLE IF NOT EXISTS trucks_main (
    id SERIAL PRIMARY KEY,
    truck_number TEXT NOT NULL UNIQUE,
    trucking_company TEXT NOT NULL DEFAULT '',
    notes TEXT,
    truck_size TEXT NOT NULL DEFAULT '',
    phone TEXT,
    license_plate TEXT,
    active BOOLEAN NOT NULL DEFAULT TRUE
);"""
)
conn.commit()
print("Table trucks_main created successfully!")

Table trucks_main created successfully!


In [ ]:
for index, row in df_materials.iterrows():
    cur.execute("""
        INSERT INTO material_price 
        (cat, material, direction, axle1, axle2, axle3, axle4, axle5, axle6, axle7, axle8, axle9)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        row['Cat'],
        row['Material new name'],
        row['Direction'],
        row['1'],
        row['2'],
        row['3'],
        row['4'],
        row['5'],
        row['6'],
        row['7'],
        row['8'],
        row['9']
    ))

conn.commit()

print("Data inserted successfully into material_price!")

In [29]:
for index, row in df_customers.iterrows():
    cur.execute("""
        INSERT INTO customers 
        (customer_name, full_address, contact_person, phone_number, notes)
        VALUES (%s, %s, %s, %s, %s)
    """, (
        row['Customer Name'],
        row['full_Address'],
        row['Contact'],
        row['Phone #'],
        row['Notes']
    ))
conn.commit()
print("Data inserted successfully into customers!")
for index, row in df_trucks.iterrows():
    axle = row["Size"]
    truck_size_label = f"axle{axle}"

    cur.execute("""
        INSERT INTO trucks_main 
        (truck_number, trucking_company, notes, truck_size, phone, license_plate)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (
        row['truck_number'],
        row['trucking_company'],
        row['Notes'],
        truck_size_label,
        row['Phone'],
        row['license_plate']
    ))
conn.commit()
print("Data inserted successfully into trucks_main!")

Data inserted successfully into customers!
Data inserted successfully into trucks_main!


In [32]:
cur.execute("""
            Select *
            FROM trucks_main""")
ans=cur.fetchall()
for i in ans:
    print(i)

(1, 'LAWN01', '44 LAWNCARE LANDSCAPING SERVICES', 'NaN', 'axle1.0', 'NaN', 'PLM7176', True)
(2, 'ACE74', 'ACE Trucking', 'AL', 'axle5.0', 'NaN', 'NaN', True)
(3, 'ACE75', 'ACE Trucking', 'NaN', 'axle5.0', 'NaN', 'NaN', True)
(4, 'ACE76', 'ACE Trucking', 'NaN', 'axle5.0', 'NaN', 'PLB7535', True)
(5, 'BEP13', 'BEP TRUCKING', 'NaN', 'axle7.0', '330-483-4252', 'TRF5008', True)
(6, 'BEP14', 'BEP TRUCKING', 'NaN', 'axle7.0', '330-483-4252', 'NaN', True)
(7, 'BEP15', 'BEP TRUCKING', 'NaN', 'axle7.0', '330-483-4252', 'NaN', True)
(8, 'BEP16', 'BEP TRUCKING', 'NaN', 'axle7.0', '330-483-4252', 'NaN', True)
(9, 'BEP24', 'BEP TRUCKING', 'NaN', 'axle7.0', '330-483-4252', 'TTM2351', True)
(10, 'BEP28', 'BEP TRUCKING', 'NaN', 'axle7.0', '330-483-4252', 'NaN', True)
(11, 'BEP7', 'BEP TRUCKING', 'NaN', 'axle7.0', '330-483-4252', 'NaN', True)
(12, 'BEP8', 'BEP TRUCKING', 'NaN', 'axle7.0', '330-483-4252', 'NaN', True)
(13, 'BUM716', 'BUMBLEBEE TRUCKING', 'NaN', 'axle5.0', 'NaN', 'NaN', True)
(14, 'BUM717

In [34]:
import pandas as pd

file_id = "1Eh6-Yzyqh7htcU2SJjeyT3i9TdcWxpHI"

url = f"https://drive.google.com/uc?id={file_id}"

df = pd.read_csv(url)

print(df.columns)

Index(['Job #', 'Job Name', 'Contract $', 'Contract Cost', 'Proj Class',
       'Job Status', 'PM ID', 'Job Type', 'Job St 1', 'Job St 2', 'Job City',
       'Job State', 'Job Zip', 'Job County', 'Job Delivery Address',
       'Tax Exempt', 'Exemption Certificate #', 'Master Job #', 'Start Year',
       'Complete Yr', 'Customer #', 'Customer Name', 'Contact',
       'Contact Email', 'Project Phone', 'Customer Job #', 'Retainage',
       'EEO Reporting', 'Hours', 'Distance from Office', 'Cert P/R',
       'Bill Type', 'MBE/EDGE', 'Customer Add1', 'Customer Add2', 'Cust City',
       'Cust State', 'Cust Zip', 'Cust Type', 'OCIP/CCIP', 'Column2',
       'Interco Cust', 'Name'],
      dtype='str')


In [1]:
import psycopg2
conn = psycopg2.connect(
    host="tktsys-db.postgres.database.azure.com",
    database="postgres",
    user="tktsys_pgdb",
    password="Mccraken_database",
    port=5432,
    sslmode="require"
)

cur = conn.cursor()
cur.execute("""
SELECT *
FROM tickets""")
for i in cur.fetchall():
    print(i)

(1, 'DT-2026-000001', 2026, 1, 'IN', '2026-03-10T12:35:20', 3527, '15162', 'VA Hospital-Radiation (15162)', 'ARKK Holdings', 10, 'BEP28', 19, 'Rubble', 1.0, 'Load', '', 'C:\\Users\\pstanam.MRX\\Downloads\\OneDrive_2026-02-23\\McCraken Ticket System\\tickets_pdf\\2026\\DT-2026-000001.pdf', <memory at 0x000001E07701BE80>)
(2, 'DT-2026-000002', 2026, 2, 'IN', '2026-03-18T13:59:18', 3532, '15173', 'VA Hospital SPD (15173)', 'Arms Trucking', 11, 'BEP7', 11, 'Broken Concrete', 1.0, 'Load', '', 'C:\\Users\\pstanam.MRX\\Downloads\\OneDrive_2026-02-23\\McCraken Ticket System\\tickets_pdf\\2026\\DT-2026-000002.pdf', <memory at 0x000001E07714C100>)


In [3]:
cur.close()
conn.close()

# Important


In [50]:
print("""Manifesting this Year\nPlease Devuda....Please Lets Do it fo the betterment of tomorrow me....\n\
"1)Jim should say 'lets move ahead with data scientist category----Done, Thank you\n"\
"2)Attorney should accept to Jim's decisison and apply from Data scientist category----Done, Thank you\n"\
"3)The Lottery has to be succesfully picked----Done, Thank you\n"\
"4)the petition has to be filed succesfully and fees submitted\n"\
"4.1)Jim should email the attorney regd the next steps of filing the petition ASAP.\n"\
"4.2)Attorney should respond wiht the next steps of filing the petition.\n"\
"4.3)Jim and I should submit ll the documents that are needed for filing and the Attorney has to submit the documents.\n"\
"4.4) The application has to be submitted succesfully and fees paid.\n"\
"5)the petition should  be reviewed and the H1B has to be approved succesfully.\n"\
"6)I should peacefully continue my work in Mr.Excavator with my H1B visa starting from Oct 2026\n"\
"7)I should work hard and grow in this role and better opportunities to come in future\n"\
"8)All the people i know should be wihth helthy lifestyle and have the best times of their life!\n"\
"9)Nanna, atta should come to US this year and i shoudl be wiht them to travel wiht me sponsering the trip.\n"\
"10)By then i should buy BMW X3 M sport edition and take them both in the car""")

Manifesting this Year
Please Devuda....Please Lets Do it fo the betterment of tomorrow me....
"1)Jim should say 'lets move ahead with data scientist category----Done, Thank you
""2)Attorney should accept to Jim's decisison and apply from Data scientist category----Done, Thank you
""3)The Lottery has to be succesfully picked----Done, Thank you
""4)the petition has to be filed succesfully and fees submitted
""4.1)Jim should email the attorney regd the next steps of filing the petition ASAP.
""4.2)Attorney should respond wiht the next steps of filing the petition.
""4.3)Jim and I should submit ll the documents that are needed for filing and the Attorney has to submit the documents.
""4.4) The application has to be submitted succesfully and fees paid.
""5)the petition should  be reviewed and the H1B has to be approved succesfully.
""6)I should peacefully continue my work in Mr.Excavator with my H1B visa starting from Oct 2026
""7)I should work hard and grow in this role and better opportun

In [43]:
rows=5
columns=5
a=[[0]*columns for _ in range(rows)]
for i in range(rows):
    for j in range(columns):
        a[i][j]=i+j
for row in a:
    print(row)    

[0, 1, 2, 3, 4]
[1, 2, 3, 4, 5]
[2, 3, 4, 5, 6]
[3, 4, 5, 6, 7]
[4, 5, 6, 7, 8]


# Interaction with Database

In [3]:
import psycopg2
import os
from dotenv import load_dotenv
load_dotenv(".env")
host = os.getenv("PGHOST", "").strip()
database = os.getenv("PGDATABASE", "").strip()
user = os.getenv("PGUSER", "").strip()
password = os.getenv("PGPASSWORD", "").strip()
conn = psycopg2.connect(
    host=host,
    database=database,
    user=user,
    password=password,
    port=5432,
    sslmode="require"
)


In [6]:
cur = conn.cursor()
Query = "SELECT * FROM jobs_cache LIMIT 0"
cur.execute(Query)

column_names = [desc[0] for desc in cur.description]

for col in column_names:
    print(col)

id
job_code
job_name
customer
active
source_updated_at
refreshed_at
tax_exempt


In [7]:
cur = conn.cursor()
Query="SELECT * FROM jobs_cache" 
cur.execute(Query) 
for i in cur.fetchall(): 
    print(i)

(2627, '6667', 'Alcosan N.End Plant Exp (6667)', 'Mascaro Construction Co., LP', 1, None, '2026-04-09T14:28:35', 'Y')
(2898, '7023', 'SHPL Bertram Wood (7023)', 'Turner Construction Company', 1, None, '2026-04-09T14:28:35', 'Y')
(2915, '7040', 'SCS Bus Lot (7040)', 'Redcon, LLC', 1, None, '2026-04-09T14:28:35', 'N')
(2635, '6692', 'Brooklyn City Center (6692)', 'Panzica Construction', 1, None, '2026-04-09T14:28:35', 'Y')
(2636, '6696', 'Nalco Water (6696)', 'The Beaver Excavating Co.', 1, None, '2026-04-09T14:28:35', 'N')
(2687, '6811', 'Mars Middle School (6811)', 'Independence Excavating', 0, None, '2026-04-09T14:28:35', 'Y')
(2688, '6812', 'Herron Hill Temp Fence (6812)', 'Redcon, LLC', 1, None, '2026-04-09T14:28:35', 'Y')
(2689, '6813', 'Geneva Memorial Field (6813)', 'Redcon, LLC', 1, None, '2026-04-09T14:28:35', 'Y')
(2690, '6814', 'Gateway Economic Dev. (6814)', 'Redcon, LLC', 0, None, '2026-04-09T14:28:35', 'Y')
(2705, '6829', 'Ohio CAT 9500 Brookpark (6829)', 'Redcon, LLC', 0,

In [19]:
cur.close()
conn.close()

NameError: name 'cur' is not defined

In [4]:
Fitness_manifestation="""My Manifestation for a Fitness Journey
1) I should become below 100 Kgs by August 15 2026
2) My veins should pop out and be visible on my arms and neck
3) I should have noticable abs (4 pack atleast)
4) I will become a toned physic with athletic personality
5) I have to be consistent with my workouts and diet in the whole process
6) I should become a person with increased stamina and endurance
7) I should include swimming as part of my routine to become fit and healthy"""

print(Fitness_manifestation)

My Manifestation for a Fitness Journey
1) I should become below 100 Kgs by August 15 2026
2) My veins should pop out and be visible on my arms and neck
3) I should have noticable abs (4 pack atleast)
4) I will become a toned physic with athletic personality
5) I have to be consistent with my workouts and diet in the whole process
6) I should become a person with increased stamina and endurance
7) I should include swimming as part of my routine to become fit and healthy


In [ ]:
# One-time backfill: populate tickets.cost from truck_size + material axle price
# Run this once after schema update.

import os
import re
import psycopg2
from dotenv import load_dotenv

load_dotenv(".env")

RECALCULATE_ALL = False  # False = only rows with cost=0/NULL, True = recompute all
DRY_RUN = False           # Set to False to apply updates


def truck_size_to_axle_index(truck_size):
    size_text = str(truck_size or "").strip().lower()
    if not size_text:
        return None

    match = re.search(r"axle\s*([0-9]+(?:\.[0-9]+)?)", size_text)
    if not match:
        return None

    axle_value = float(match.group(1))
    axle_index = int(axle_value)
    if axle_value != axle_index:
        return None

    return axle_index if 1 <= axle_index <= 9 else None


def create_connection():
    database_url = os.getenv("DATABASE_URL", "").strip()
    if database_url:
        return psycopg2.connect(database_url)

    return psycopg2.connect(
        host=os.getenv("PGHOST", "").strip(),
        port=int(os.getenv("PGPORT", "5432").strip()),
        dbname=os.getenv("PGDATABASE", "").strip(),
        user=os.getenv("PGUSER", "").strip(),
        password=os.getenv("PGPASSWORD", "").strip(),
        sslmode=os.getenv("PGSSLMODE", "require").strip(),
    )


conn = create_connection()
conn.autocommit = False

try:
    with conn.cursor() as cursor:
        cursor.execute("ALTER TABLE tickets ADD COLUMN IF NOT EXISTS cost REAL NOT NULL DEFAULT 0")

        where_clause = "" if RECALCULATE_ALL else "WHERE COALESCE(t.cost, 0) = 0"
        cursor.execute(
            f"""
            SELECT
                t.id,
                COALESCE(t.quantity, 0) AS quantity,
                tm.truck_size,
                mp.axle1, mp.axle2, mp.axle3, mp.axle4, mp.axle5,
                mp.axle6, mp.axle7, mp.axle8, mp.axle9
            FROM tickets t
            LEFT JOIN trucks_main tm ON tm.id = t.truck_id
            LEFT JOIN material_price mp ON mp.id = t.material_id
            {where_clause}
            ORDER BY t.id
            """
        )
        rows = cursor.fetchall()
        print(rows)

    updates = []
    skipped = 0

    for row in rows:
        (
            ticket_id,
            quantity,
            truck_size,
            axle1,
            axle2,
            axle3,
            axle4,
            axle5,
            axle6,
            axle7,
            axle8,
            axle9,
        ) = row

        axle_index = truck_size_to_axle_index(truck_size)
        print(f"Ticket ID: {ticket_id}, Truck Size: {truck_size}, Axle Index: {axle_index}")
        if not axle_index:
            skipped += 1
            continue

        axle_values = {
            1: axle1,
            2: axle2,
            3: axle3,
            4: axle4,
            5: axle5,
            6: axle6,
            7: axle7,
            8: axle8,
            9: axle9,
        }

        price_per_load = float(axle_values.get(axle_index) or 0)
        print(f"Price per load for axle {axle_index}: {price_per_load}")
        cost = round(float(quantity or 0) * price_per_load, 2)
        print(f"Calculated cost for ticket {ticket_id}: {cost}")    
        updates.append((cost, ticket_id))

    print(f"Rows selected: {len(rows)}")
    print(f"Rows with calculable cost: {len(updates)}")
    print(f"Rows skipped (missing/invalid truck size or price): {skipped}")

    if DRY_RUN:
        print("DRY_RUN=True, no database updates were applied.")
        conn.rollback()
    else:
        with conn.cursor() as cursor:
            cursor.executemany("UPDATE tickets SET cost = %s WHERE id = %s", updates)
        conn.commit()
        print(f"Updated rows: {len(updates)}")

except Exception as exc:
    conn.rollback()
    raise
finally:
    conn.close()

[(1, 1.0, 'axle7.0', 75.0, 90.0, 120.0, 165.0, 165.0, 180.0, 195.0, nan, nan), (2, 1.0, 'axle7.0', 75.0, 90.0, 120.0, 165.0, 165.0, 180.0, 195.0, nan, nan), (3, 1.0, 'axle5.0', 80.0, 96.0, 128.0, 176.0, 176.0, 192.0, 208.0, nan, nan), (4, 1.0, 'axle5.0', 80.0, 96.0, 128.0, 176.0, 176.0, 192.0, 208.0, nan, nan), (5, 1.0, 'axle5.0', 80.0, 96.0, 128.0, 176.0, 176.0, 192.0, 208.0, nan, nan), (6, 1.0, 'axle5.0', None, None, None, None, None, None, None, None, None), (7, 1.0, 'axle5.0', 125.0, 150.0, 200.0, 275.0, 275.0, 300.0, 325.0, nan, nan), (9, 1.0, 'axle5.0', 30.0, 36.0, 48.0, 66.0, 66.0, 72.0, 78.0, nan, nan), (10, 1.0, 'axle5.0', None, None, None, None, None, None, None, None, None), (11, 1.0, 'axle5.0', nan, nan, nan, nan, nan, nan, nan, nan, nan), (12, 1.0, 'axle7.0', 625.0, 750.0, 1000.0, 1375.0, 1375.0, 1500.0, 1625.0, nan, nan), (13, 1.0, 'axle5.0', None, None, None, None, None, None, None, None, None), (14, 2.0, 'axle5.0', None, None, None, None, None, None, None, None, None), 

In [54]:
import psycopg2
import os
from dotenv import load_dotenv
load_dotenv(".env")
host = os.getenv("PGHOST", "").strip()
database = os.getenv("PGDATABASE", "").strip()
user = os.getenv("PGUSER", "").strip()
password = os.getenv("PGPASSWORD", "").strip()
conn = psycopg2.connect(
    host=host,
    database=database,
    user=user,
    password=password,
    port=5432,
    sslmode="require"
)


In [17]:
import pandas as pd
import re
csv_path = r"C:\Users\pstanam.MRX\Downloads\OneDrive_2026-02-23\McCraken Ticket System\Copy of 2025 DumpLog  Use This Log per TAF.csv"
df_csv = pd.read_csv(csv_path)
df_csv['Date'] = pd.to_datetime(df_csv['Date'], errors='coerce')
df_csv = df_csv.loc[:, ~df_csv.columns.str.contains(r"^Unnamed", case=False)]
job_raw = df_csv["Job Name"]
job_text = job_raw.fillna("").astype(str)
prefix_pattern = r"^\s*\(?\s*(IN|OUT)\s*\)?\s*"
df_csv["direction_from_name"] = (
    job_text.str.extract(prefix_pattern, flags=re.IGNORECASE)[0]
    .str.upper()
)
df_csv["job_name_clean"] = (
    job_text.str.replace(prefix_pattern, "", regex=True, flags=re.IGNORECASE)
    .str.strip()
)
df_csv.loc[job_raw.isna(), ["direction_from_name", "job_name_clean"]] = pd.NA
df_csv["job code"] = (
    df_csv["job_name_clean"]
    .fillna("")
    .str.extract(r"\((\d+)\)\s*$", expand=False)
    .fillna("")
)
pattern = r"\s*\((IN|OUT)\)\s*$"

# 1) On df_csv
df_csv["direction_from_material"] = (
    df_csv["Material2"].fillna("")
    .astype("str")
    .str.extract(pattern, expand=False ,flags=re.IGNORECASE)
    .str.upper()
)

df_csv["material_clean"] = (
    df_csv["Material2"].fillna("")
    .astype("str")
    .str.replace(pattern, "", regex=True, flags=re.IGNORECASE)
    .str.strip()
    .str.lower()
)

In [18]:
# Build final_direction from the two direction columns
col_name_dir="direction_from_name"
col_mat_dir="direction_from_material"
d1_final = (
    df_csv[col_name_dir]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .str.upper()
)

d2_final = (
    df_csv[col_mat_dir]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .str.upper()
)

df_csv["final_direction"] = pd.Series(pd.NA, index=df_csv.index, dtype="string")

only_d1 = d1_final.notna() & d2_final.isna()
only_d2 = d1_final.isna() & d2_final.notna()
both_same = d1_final.notna() & d2_final.notna() & (d1_final == d2_final)
both_diff = d1_final.notna() & d2_final.notna() & (d1_final != d2_final)

df_csv.loc[only_d1, "final_direction"] = d1_final[only_d1]
df_csv.loc[only_d2, "final_direction"] = d2_final[only_d2]
df_csv.loc[both_same, "final_direction"] = d1_final[both_same]
df_csv.loc[both_diff, "final_direction"] = "DIFF"

In [19]:
df_csv["final_direction"] = df_csv.apply(lambda row: "IN" if row['material_clean'] == "Wet Dirt w/ Organics" else row["final_direction"], axis=1)

In [20]:
df_csv2=df_csv.copy()
df_csv2['Year'] = df_csv2['Date'].dt.year
df_csv2["date_time"] = pd.to_datetime(
    df_csv2["Date"].astype(str).str.strip() + " " + df_csv2["Time"].astype(str).str.strip(),
    format="%Y-%m-%d %I:%M %p",
    errors="coerce"
).dt.strftime("%Y-%m-%dT%H:%M:%S")
df_csv2.drop(columns=["Material", "Material2","direction_from_name","direction_from_material","Tick-Date-Mat-Size","Customer","Job Name"], inplace=True)
df_csv2["Truck #_clean"] = (
    df_csv2["Truck #"]
    .astype(str)
    .str.strip()
    .str.lower()
)


In [21]:
df_csv_2026 = df_csv2[df_csv2['Date'].dt.year == 2026]
df_csv_tillmarch2026 = df_csv_2026[df_csv_2026['Date'].dt.month <= 3]


In [22]:
df_csv_tillmarch2026.to_csv(r"C:\Users\pstanam.MRX\Downloads\OneDrive_2026-02-23\McCraken Ticket System\processed_till_march2026.csv", index=False)

In [24]:
mapping = {
    "1 Axle": "Axle 1",
    "2 Axle": "Tandem",
    "3 Axle": "TriAxle",
    "4/5 Axle": "Axle 4_5",
    "6 Axle": "Axle 6",
    "7 Axle": "Semi",
    "8 Axle": "HydVac",
    "9 Axle": "DIRT_IN"
}

df_csv_tillmarch2026['Trucksize'] = df_csv_tillmarch2026['Trucksize'].replace(mapping)

In [ ]:
df

In [58]:
df_jobs_cache = pd.read_sql_query("SELECT * FROM jobs_cache", conn)
df_jobs_cache = df_jobs_cache.replace({"NaN": pd.NA, "nan": pd.NA})
df_tickets = pd.read_sql_query("SELECT * FROM tickets", conn)
df_tickets = df_tickets.replace({"NaN": pd.NA, "nan": pd.NA})
df_manualjobs = pd.read_sql_query("SELECT * FROM manual_jobs", conn)
df_manualjobs = df_manualjobs.replace({"NaN": pd.NA, "nan": pd.NA})
df_trucks = pd.read_sql_query("SELECT * FROM trucks_main", conn)
df_trucks = df_trucks.replace({"NaN": pd.NA, "nan": pd.NA})
df_materials = pd.read_sql_query("SELECT * FROM material_price", conn)
df_materials = df_materials.replace({"NaN": pd.NA, "nan": pd.NA})
df_customers = pd.read_sql_query("SELECT * FROM customers", conn)
df_customers = df_customers.replace({"NaN": pd.NA, "nan": pd.NA})
df_trucks["truck_number_clean"] = (
    df_trucks["truck_number"]
    .astype(str)
    .str.strip()
    .str.lower()
)
df_materials["material_clean"] = (
    df_materials["material"]
    .astype(str)
    .str.strip()
    .str.lower()
)

C:\Users\pstanam.MRX\AppData\Local\Temp\ipykernel_32152\671322437.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_jobs_cache = pd.read_sql_query("SELECT * FROM jobs_cache", conn)
C:\Users\pstanam.MRX\AppData\Local\Temp\ipykernel_32152\671322437.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tickets = pd.read_sql_query("SELECT * FROM tickets", conn)
C:\Users\pstanam.MRX\AppData\Local\Temp\ipykernel_32152\671322437.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_manualjobs = pd.read_sql_query("SELECT * F

In [59]:
df_manualjobs.rename(columns={"created_at": "source_updated_at"}, inplace=True)
df_alljobs = pd.concat([
    df_jobs_cache[["job_code", "job_name", "customer", "tax_exempt", "active", "source_updated_at"]],
    df_manualjobs[["job_code", "job_name", "customer", "tax_exempt", "active", "source_updated_at"]]
], ignore_index=True)

In [ ]:
df_csv_tillmarch2026[df_csv_tillmarch2026["job code"].notna() & (df_csv_tillmarch2026["job code"] != "")][["job code", "job_name_clean", "Customer Name", "final_direction", "date_time"]].head()

,id,job_code,job_name,customer,active,source_updated_at,refreshed_at,tax_exempt
0,2627,6667,Alcosan N.End Plant Exp (6667),"Mascaro Construction Co., LP",1,None,2026-04-27T13:38:37,Y
1,2732,6857,Waltco Parking Lot (6857),"Redcon, LLC",1,None,2026-04-27T13:38:37,N
2,2808,6933,Lucasville Sallyport (6933),"Redcon, LLC",1,None,2026-04-27T13:38:37,N
3,2837,6962,Novak CF Gate (6962),"Redcon, LLC",1,None,2026-04-27T13:38:37,N
4,3238,2224,Education First C.U. (2224),Lincoln Construction,0,None,2026-04-27T13:38:37,N


In [60]:
df_merged = df_csv_tillmarch2026.merge(df_jobs_cache, left_on="job code", right_on="job_code", how="left", suffixes=("", "_job"))
df_merged = df_merged.merge(df_trucks, left_on="Truck #_clean", right_on="truck_number_clean", how="left", suffixes=("", "_truck"))
df_merged = df_merged.merge(df_materials, left_on="material_clean", right_on="material_clean", how="left", suffixes=("", "_mat"))
df_merged = df_merged.merge(df_customers, left_on="Customer Name", right_on="customer_name", how="left", suffixes=("", "_cust"))

In [62]:
df_merged['tax_exempt'] = df_merged['tax_exempt'].fillna('N')

In [64]:
cols = [
    "Ticket #","date_time", " $ ", "Customer Name",
    "job_name_clean", "job code", "material_clean",
    "final_direction", "Year", "Truck #_clean",
    "tax_exempt", "id_truck", "id_mat", "material","id_cust","Trucksize"
]

df_merged = df_merged[cols]

In [65]:
df_merged['Trucksize'].value_counts(dropna=False)

Trucksize
Axle 4_5    1924
Axle 1       275
TriAxle      215
Hydrovac      80
Semi          78
Tandem        52
Name: count, dtype: int64

In [67]:
mask = df_merged['job code'].astype(str).str.strip() == ""

df_merged.loc[mask, 'job code'] = df_merged.loc[mask, 'job_name_clean']

In [70]:
df_merged['Truck #_clean'] = df_merged['Truck #_clean'].str.upper()
df_merged.head()

,Ticket #,date_time,$,Customer Name,job_name_clean,job code,material_clean,final_direction,Year,Truck #_clean,tax_exempt,id_truck,id_mat,material,id_cust,Trucksize
0,170376,2026-01-02T09:58:00,$75.00,"Middleton Mechanical, LLC",Parkview Dr.,Parkview Dr.,other clean hard fill,IN,2026,MID902,N,110,18.0,Other Clean Hard Fill,87,Axle 1
1,170377,2026-01-05T07:49:00,$165.00,Petty Group LLC,Woodhill Roads PH1 (2720),2720,other clean hard fill,IN,2026,DEV5,N,54,18.0,Other Clean Hard Fill,106,Axle 4_5
2,170378,2026-01-05T08:08:00,$165.00,Petty Group LLC,Woodhill Roads PH1 (2720),2720,other clean hard fill,IN,2026,DEV3,N,52,18.0,Other Clean Hard Fill,106,Axle 4_5
3,170379,2026-01-05T08:19:00,$120.00,Mike Coates Construction Co.,RTA,RTA,other clean hard fill,IN,2026,MIKE108,N,205,18.0,Other Clean Hard Fill,88,TriAxle
4,170380,2026-01-05T08:26:00,$165.00,Petty Group LLC,Woodhill Roads PH1 (2720),2720,other clean hard fill,IN,2026,DC33,N,38,18.0,Other Clean Hard Fill,106,Axle 4_5


In [56]:
cursor = conn.cursor()

insert_query = """
INSERT INTO manual_jobs (job_code, job_name, customer, tax_exempt, active, created_at)
VALUES (%s, %s, %s, %s, %s, %s)
ON CONFLICT (job_code, job_name) DO NOTHING;
"""

for _, row in df_merged_nojobcode.iterrows():
    cursor.execute(insert_query, (
        row["job_name_clean"],
        row["job_name_clean"],
        row["Customer Name"],
        'N',
        1,  
        row["date_time"]
    ))

conn.commit()
cursor.close()

In [98]:
cursor = conn.cursor()

insert_query = """
INSERT INTO trucks_main (truck_number, trucking_company, truck_size, active)
SELECT %s, %s, %s, %s
WHERE NOT EXISTS (
    SELECT 1 FROM trucks_main WHERE truck_number = %s
)
"""

for _, row in df_missing_truck_id.iterrows():
    cursor.execute(insert_query, (
        row["Truck #_clean"],
        "Legacy Data (Unknown Hauler)",
        row["Trucksize"],
        True,  
        row["Truck #_clean"]
    ))

conn.commit()
cursor.close()

In [50]:
df_merged_nojobcode = df_merged[df_merged['job code'].astype(str).str.strip() == ""]

df_merged_nojobcode = df_merged_nojobcode[
    ['job_name_clean','job code','Customer Name','tax_exempt','date_time']
].drop_duplicates(subset=['job_name_clean', 'Customer Name'])

In [51]:
df_merged_nojobcode.head()

,job_name_clean,job code,Customer Name,tax_exempt,date_time
0,Parkview Dr.,,"Middleton Mechanical, LLC",NaN,2026-01-02T09:58:00
3,RTA,,Mike Coates Construction Co.,NaN,2026-01-05T08:19:00
8,Bayberry Rd. - Northfield,,Don Wartko Construction,NaN,2026-01-05T09:13:00
20,Lee Rd.,,Utilities Construction Co.,NaN,2026-01-05T10:01:00
73,5136 N. Ridge Rd.,,Du-All Services,NaN,2026-01-06T10:29:00


In [46]:
df_merged_nojobcode['job_name_clean'].value_counts(dropna=False)

job_name_clean
Lee Rd.                                     2
Parkview Dr.                                1
RTA                                         1
Bayberry Rd. - Northfield                   1
5136 N. Ridge Rd.                           1
Bedford - Solon Rd.                         1
Credit- CMG Contracting- Meadow Park        1
Credit - Badger - Parma                     1
Wamelink Ave. - Woodhill Station            1
Bayberry Rd.-Northfield                     1
JCU                                         1
Shaker Hts.                                 1
Madison Ave - Lakewood                      1
93rd & Wamelink Ave. - Woodhill Station     1
Job #3853 Edgepark                          1
Bedford Hts.                                1
Edgepark Dr.                                1
Brecksville Rd.                             1
Parkview Dr. - Job #4058                    1
Job #3853                                   1
Credit - Dozer Enterprise - Pavlowski       1
Credit - O.M. Const

C:\Users\pstanam.MRX\AppData\Local\Temp\ipykernel_32152\3126921066.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_manualjobs = pd.read_sql_query("SELECT * FROM manual_jobs", conn)


,id,job_code,job_name,customer,tax_exempt,active,created_at
0,1,wendys job,wendys job,,New,1,2026-04-10T10:15:56
1,2,new job 54,new job 54,,New,1,2026-04-10T10:20:21
2,36,Parkview Dr.,Parkview Dr.,"Middleton Mechanical, LLC",N,1,2026-01-02T09:58:00
3,37,RTA,RTA,Mike Coates Construction Co.,N,1,2026-01-05T08:19:00
4,38,Bayberry Rd. - Northfield,Bayberry Rd. - Northfield,Don Wartko Construction,N,1,2026-01-05T09:13:00


In [48]:
cur = conn.cursor()

query = "DELETE FROM manual_jobs WHERE id = %s;"
cur.execute(query, (3,))

conn.commit()

cur.close()
conn.close()

In [113]:
df_merged_nojobcode['job_name_clean'].value_counts(dropna=False)

job_name_clean
Parkview Dr.                                108
Lee Rd.                                      99
Bedford                                      78
Job #3853                                    69
RTA                                          56
VA Hospital - 3311 W.73rd St.                54
Brecksville                                  47
Bedford - Solon Rd.                          31
JCU                                          31
Parkview Dr. - Job #4058                     31
Credit - Dozer Enterprise - Pavlowski        27
93rd & Buckeye                               14
Day Dr. - Parma                              10
Yard - Woodhill                              10
Shaker Hts.                                   9
VA Hospital - 331 W.73rd St.                  8
Bayberry Rd.-Northfield                       7
93rd & Wamelink Ave. - Woodhill Station       7
Credit-Ready Rock Concrete                    7
Job #3853 Edgepark                            5
Bedford Hts.             

In [61]:
df_materials.rename(columns = { 'Rev# 11\n1 Axle': 'Axle 1', 'Rev# 11\nTandem': 'Tandem', 'Rev# 11\nTriAxle': 'TriAxle', 'Rev# 11\n4/5 Axle': 'Axle 4_5', 'Rev# 11\n6 Axle': 'Axle 6', 'Rev# 11\nSemi': 'Semi', 'Rev# 11\nHydVac': 'HydVac', 'Rev# 11\nDIRT IN': 'DIRT IN' }, inplace = True)
truck_size_map = {
    "axle1.0": "Axle 1",
    "axle1": "Axle 1",
    "axle2.0": "Tandem",
    "axle2": "Tandem",
    "axle3.0": "TriAxle",
    "axle3": "TriAxle",
    "axle4.0": "Axle 4_5",
    "axle4": "Axle 4_5",
    "axle5.0": "Axle 4_5",
    "axle5": "Axle 4_5",
    "axle6.0": "Axle 6",
    "axle6": "Axle 6",
    "axle7.0": "Semi",
    "axle7": "Semi",
    "axle8.0": "HydVac",
    "axle8": "HydVac",
    "axle9.0": "DIRT IN",
    "axle9": "DIRT IN",
}

df_trucks_main["truck_size_new"] = (
    df_trucks_main["truck_size"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace(truck_size_map)
)

df_trucks_main["truck_size_new"].value_counts(dropna=False)

truck_size_new
Axle 4_5    111
Axle 1       40
HydVac       16
TriAxle      15
Semi         13
Tandem        7
Name: count, dtype: Int64

In [67]:
df_csv_2026 = df_csv[df_csv['Date'].dt.year == 2026]
df_csv_jan2026 = df_csv_2026[df_csv_2026['Date'].dt.month == 1]

In [68]:
unique_jan_jobs = list(df_csv_jan2026['Job Name'].unique())
unique_jan_jobs_clean = list(df_csv_jan2026['job_name_clean'].unique())
print(unique_jan_jobs)
print(f"Total unique jan jobs: {len(unique_jan_jobs)}")
print(unique_jan_jobs_clean)
print(f"Total unique jan jobs (cleaned): {len(unique_jan_jobs_clean)}")

['Parkview Dr. ', 'Woodhill Roads PH1 (2720)', 'RTA', 'OUT Woodhill Roads PH1 (2720)', 'OUT Bayberry Rd. - Northfield', 'OUT  Woodhill Roads PH1 (2720)', 'Lee Rd.', 'W. 26th Apartments (2743)', 'OUT Parkview Dr. ', 'OUT  Parkview Dr. ', '(OUT) Parkview Dr.', '5136 N. Ridge Rd.', 'Foster Senior Lofts (15306)', 'OUT Heritage-Hanna Cottage (15341)', 'Riverview Elementary (7497)', 'Bedford - Solon Rd.', 'Credit- CMG Contracting- Meadow Park', 'Credit - Badger - Parma', 'Heritage-Hanna Cottage (15341)', 'Wamelink Ave. - Woodhill Station', 'Bayberry Rd.-Northfield', 'OUT Bayberry Rd.-Northfield', 'Canterbury Golf Club (2748)', 'JCU', 'Misc. Hydro-Vac CLE 2026 (2762)', 'Shaker Hts.', 'Madison Ave - Lakewood', 'CLE Zoo Perimeter (7355)', '93rd & Wamelink Ave. - Woodhill Station', 'Beechmont Country Club (15339)', 'Job #3853 Edgepark', 'Northern Ohio Blanket Mills (7476)', 'OUT Job #3853 Edgepark', 'Bedford Hts.', 'Edgepark Dr.', 'Brecksville Rd.', 'OUT Brecksville Rd. ', 'Kenjoh Outdoor (2771)

In [72]:
cur = conn.cursor()
cur.execute("SELECT DISTINCT job_code FROM jobs_cache")
rows = cur.fetchall()
job_codes = [r[0] for r in rows if r and r[0] is not None]
cur.execute("SELECT DISTINCT job_code FROM jobs_cache")
rows = cur.fetchall()
job_codes = [r[0] for r in rows if r and r[0] is not None]
 


['15101', '15103', '15114', '15116', '15126', '15140', '15141', '15142', '15143', '15144', '15145', '15146', '15147', '15148', '15149', '15150', '15151', '15152', '15153', '15154', '15155', '15156', '15157', '15158', '15159', '15160', '15161', '15162', '15163', '15164', '15165', '15166', '15167', '15168', '15169', '15170', '15171', '15172', '15173', '15174', '15175', '15176', '15177', '15178', '15179', '15180', '15181', '15182', '15183', '15184', '15185', '15186', '15187', '15188', '15189', '15190', '15191', '15192', '15193', '15194', '15195', '15196', '15197', '15198', '15199', '15200', '15201', '15202', '15203', '15204', '15205', '15206', '15207', '15208', '15209', '15210', '15211', '15212', '15213', '15214', '15215', '15216', '15217', '15218', '15219', '15220', '15221', '15222', '15223', '15224', '15225', '15226', '15227', '15228', '15229', '15230', '15231', '15232', '15233', '15234', '15235', '15236', '15237', '15238', '15239', '15241', '15242', '15243', '15244', '15245', '15246', 

In [78]:
jobs_to_check = (
    df_csv2[["job_name_clean", "job code"]]
    .drop_duplicates()
    .fillna("")
)
display(jobs_to_check)

,job_name_clean,job code
0,St.Clair,
1,Primate Forest (15290),15290
8,Lakewood,
9,CCF Innovation Tower PH2 (2548),2548
10,Great Lakes Cheese,
...,...,...
24057,Job - 573.01,
24082,Columbia Gas,
24113,Chardon,
24122,2025 Woodhill,


In [74]:
# 1) Prepare unique jobs from CSV (name + code)
jobs_to_check = (
    df_csv2[["job_name_clean", "job code"]]
    .drop_duplicates()
    .fillna("")
)

def normalize_text(x):
    x = str(x or "").strip().lower()
    x = re.sub(r"\s+", " ", x)
    return x

# Optional: extract trailing code like "... (15290)" if job code column is blank
def extract_code_from_name(name):
    m = re.search(r"\((\d+)\)\s*$", str(name or "").strip())
    return m.group(1) if m else ""

# 2) Build lookup sets from DB lists already available in notebook
job_names_norm_set = {normalize_text(x) for x in job_names if x is not None}
job_codes_norm_set = {normalize_text(x) for x in job_codes if x is not None}

matched_by_name = []
matched_by_code = []
missing_jobs = []

for _, r in jobs_to_check.iterrows():
    name_raw = r["job_name_clean"]
    code_raw = r["job code"]

    name_norm = normalize_text(name_raw)
    code_norm = normalize_text(code_raw) or normalize_text(extract_code_from_name(name_raw))

    if name_norm and name_norm in job_names_norm_set:
        matched_by_name.append((name_raw, code_raw))
    elif code_norm and code_norm in job_codes_norm_set:
        matched_by_code.append((name_raw, code_raw))
    else:
        missing_jobs.append((name_raw, code_raw))

print(f"Total unique CSV jobs checked: {len(jobs_to_check)}")
print(f"Matched by exact job name: {len(matched_by_name)}")
print(f"Matched by job code fallback: {len(matched_by_code)}")
print(f"Still missing: {len(missing_jobs)}")

if missing_jobs:
    print("\nMissing items:")
    for name_raw, code_raw in missing_jobs:
        suggestions = difflib.get_close_matches(str(name_raw), job_names, n=3, cutoff=0.75)
        print(f"- name='{name_raw}', code='{code_raw}'")
        if suggestions:
            print(f"  Possible name matches: {suggestions}")

Total unique CSV jobs checked: 511
Matched by exact job name: 74
Matched by job code fallback: 55
Still missing: 382

Missing items:
- name='St.Clair', code=''
- name='Lakewood', code=''
- name='Great Lakes Cheese', code=''
  Possible name matches: ['Great Lakes Cheese (6865)', 'Great Lakes Cheese (2551)', 'Great Lakes Cheese (2391)']
- name='Jr. House', code=''
- name='Rockside Rd.', code=''
- name='Cleveland Clinic', code=''
- name='Rock & Roll Hall of Fame', code=''
  Possible name matches: ['Rock & Roll Hall of Fame(2577)']
- name='Summit-Lakewood', code=''
- name='Broadview Hts.', code=''
- name='Cleveland', code=''
- name='Edgewater Park', code=''
- name='W.65th & Denison', code=''
- name='Johnston Pkwy.', code=''
- name='Yard', code=''
- name='Cedar Rd.', code=''
- name='Brooklyn Hts.', code=''
- name='E. 72nd/Union', code=''
- name='Shop', code=''
- name='Cleveland Hts.', code=''
- name='West Clifton', code=''
- name='Police Station - Fulton Rd.', code=''
- name='MGI - 130th St

In [26]:
cursor=conn.cursor()    
cursor.execute("""
SELECT * from trucks_main LIMIT 5
""")
rows = cursor.fetchall()
for r in rows:
    print(r)

(1, 'LAWN01', '44 LAWNCARE LANDSCAPING SERVICES', 'NaN', 'axle1.0', 'NaN', 'PLM7176', True)
(3, 'ACE75', 'ACE Trucking', 'NaN', 'axle5.0', 'NaN', 'NaN', True)
(4, 'ACE76', 'ACE Trucking', 'NaN', 'axle5.0', 'NaN', 'PLB7535', True)
(5, 'BEP13', 'BEP TRUCKING', 'NaN', 'axle7.0', '330-483-4252', 'TRF5008', True)
(6, 'BEP14', 'BEP TRUCKING', 'NaN', 'axle7.0', '330-483-4252', 'NaN', True)


In [9]:
import psycopg2
import os
from dotenv import load_dotenv
load_dotenv(".env")
host = os.getenv("PGHOST", "").strip()
database = os.getenv("PGDATABASE", "").strip()
user = os.getenv("PGUSER", "").strip()
password = os.getenv("PGPASSWORD", "").strip()
conn = psycopg2.connect(
    host=host,
    database=database,
    user=user,
    password=password,
    port=5432,
    sslmode="require"
)


In [10]:
cur = conn.cursor()
query = """
UPDATE customers
SET customer_name = %s
WHERE id = %s;
"""

cur.execute(query, ("InfraSource Construction LLC.", 150))

conn.commit()

cur.close()
conn.close()

In [72]:
df_merged.to_csv(r"C:\Users\pstanam.MRX\Downloads\OneDrive_2026-02-23\McCraken Ticket System\final_merged.csv", index=False)

In [94]:
df_trucks = pd.read_sql_query("SELECT * FROM trucks_main", conn)
df_trucks = df_trucks.replace({"NaN": pd.NA, "nan": pd.NA})
df_trucks.head()

C:\Users\pstanam.MRX\AppData\Local\Temp\ipykernel_27780\3416769383.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_trucks = pd.read_sql_query("SELECT * FROM trucks_main", conn)


,id,truck_number,trucking_company,notes,truck_size,phone,license_plate,active
0,1,LAWN01,44 LAWNCARE LANDSCAPING SERVICES,NaN,Axle 1,NaN,PLM7176,True
1,93,KBRJ348,KBRJ LOGISTICS LLC,NaN,Axle 4_5,NaN,PLD4450,True
2,185,ABCD1234,some dummy company,its a dummy truck 2,TriAxle,NaN,acd34,True
3,186,GAN04,GANIM COMPANY,,Axle 1,NaN,PIE4604,True
4,187,GAN08,GANIM COMPANY,,Axle 1,NaN,PJU1588,True


In [76]:
len(df_merged)

2624

In [77]:
# Backdated tickets import starter (safe by default)

# 1) Set DRY_RUN=True and confirm row counts first.

# 2) Update SOURCE_FILE and COL_MAP to match your file columns.

# 3) Set DRY_RUN=False to perform inserts.



import os

import re

from datetime import datetime



import pandas as pd

import psycopg2

from dotenv import load_dotenv



# -------------------------

# Config

# -------------------------

DRY_RUN = False

SOURCE_FILE = r"C:\Users\pstanam.MRX\Downloads\OneDrive_2026-02-23\McCraken Ticket System\final_merged.csv"



# Map your source columns to canonical names used by this script.

# Change only the values on the right side.

COL_MAP = {

    "ticket_number": "Ticket #",

    "created_at": "date_time",          # ISO datetime preferred; fallback combines Date + Time if present

    "direction": "final_direction",     # IN/OUT

    "job_code": "job code",

    "job_name": "job_name_clean",

    "customer": "Customer Name",

    "tax_exempt": "tax_exempt",

    "truck_number": "Truck #_clean",

    "material_name": "material_clean",  # or "Material2"

    "quantity": "Qty",                  # default=1 when missing

    "unit": "Unit",                     # default=Load when missing

    "cost": " $ ",

    "notes": "Notes",

}



# -------------------------

# Helpers

# -------------------------

def clean_text(value):

    if pd.isna(value):

        return ""

    return str(value).strip()



def clean_key(value):

    return re.sub(r"\s+", " ", clean_text(value)).strip().lower()



def parse_created_at(row):

    created_col = COL_MAP.get("created_at")

    created_raw = clean_text(row.get(created_col, "")) if created_col else ""

    if created_raw:

        dt = pd.to_datetime(created_raw, errors="coerce")

        if pd.notna(dt):

            return dt.to_pydatetime().replace(microsecond=0)



    # Fallback if file has separate Date and Time columns.
    date_raw = clean_text(row.get("Date", ""))
    time_raw = clean_text(row.get("Time", ""))
    combo = f"{date_raw} {time_raw}".strip()
    dt = pd.to_datetime(combo, errors="coerce")
    if pd.notna(dt):
        return dt.to_pydatetime().replace(microsecond=0)
    return datetime.now().replace(microsecond=0)
def to_float(value, default):
    txt = clean_text(value)
    if not txt:
        return default
    txt = txt.replace("$", "").replace(",", "")
    try:
        return float(txt)
    except ValueError:
        return default
def normalize_direction(value):
    direction = clean_text(value).upper()
    return direction if direction in {"IN", "OUT"} else "IN"
def parse_ticket_seed(value):
    # Expected format: DT-YYYY-000123
    ticket_no = clean_text(value)
    m = re.match(r"^DT-(\d{4})-(\d{1,})$", ticket_no)
    if not m:
        return None, None, ""
    return int(m.group(1)), int(m.group(2)), ticket_no
def ensure_ticket_sequence(cursor, year):
    cursor.execute("SELECT last_value FROM ticket_sequence WHERE ticket_year = %s", (year,))
    row = cursor.fetchone()
    if row:
        return int(row[0])
    cursor.execute(
        "INSERT INTO ticket_sequence (ticket_year, last_value) VALUES (%s, %s)",
        (year, 0),
    )
    return 0
def bump_ticket_sequence(cursor, year, last_value):
    cursor.execute(
        "UPDATE ticket_sequence SET last_value = %s WHERE ticket_year = %s",
        (last_value, year),
    )
def allocate_ticket_number(cursor, created_dt, incoming_ticket_number):
    parsed_year, parsed_seq, parsed_ticket = parse_ticket_seed(incoming_ticket_number)
    if parsed_ticket:
        year = parsed_year
        seq = parsed_seq
        ticket_number = parsed_ticket
        ensure_ticket_sequence(cursor, year)
        cursor.execute("SELECT last_value FROM ticket_sequence WHERE ticket_year = %s", (year,))
        current_last = int(cursor.fetchone()[0])
        if seq > current_last:
            bump_ticket_sequence(cursor, year, seq)
        return ticket_number, year, seq
    year = created_dt.year
    current_last = ensure_ticket_sequence(cursor, year)
    seq = current_last + 1
    bump_ticket_sequence(cursor, year, seq)
    ticket_number = f"DT-{year}-{seq:06d}"
    return ticket_number, year, seq
def build_lookup_maps(cursor):
    cursor.execute("SELECT id, truck_number FROM trucks_main")
    trucks = {clean_key(r[1]): int(r[0]) for r in cursor.fetchall() if clean_text(r[1])}
    cursor.execute("SELECT id, customer_name FROM customers")
    customers = {clean_key(r[1]): int(r[0]) for r in cursor.fetchall() if clean_text(r[1])}
    cursor.execute("SELECT id, material, direction FROM material_price")
    materials = {}
    for material_id, material_name, direction in cursor.fetchall():
        name_key = clean_key(material_name)
        dir_key = clean_text(direction).upper()
        materials[(name_key, dir_key)] = int(material_id)
        materials[(name_key, "*")] = int(material_id)
    cursor.execute("SELECT id, job_code, job_name, customer, tax_exempt FROM jobs_cache")
    jobs = []
    for row in cursor.fetchall():
        jobs.append({
            "id": int(row[0]),
            "job_code": clean_text(row[1]),
            "job_name": clean_text(row[2]),
            "customer": clean_text(row[3]),
            "tax_exempt": clean_text(row[4]),
        })
    return trucks, customers, materials, jobs
def find_job(jobs, job_code, job_name):
    code = clean_key(job_code)
    name = clean_key(job_name)
    for j in jobs:
        if code and clean_key(j["job_code"]) == code:
            return j
    for j in jobs:
        if name and clean_key(j["job_name"]) == name:
            return j
    return None
# -------------------------
# Load file
# -------------------------
if SOURCE_FILE.lower().endswith(".csv"):
    df_src = pd.read_csv(SOURCE_FILE)
else:
    df_src = pd.read_excel(SOURCE_FILE)
df_src = df_src.loc[:, ~df_src.columns.astype(str).str.contains(r"^Unnamed", case=False)]
print(f"Loaded rows: {len(df_src)}")
# -------------------------
# DB insert
# -------------------------
load_dotenv(".env")
conn = psycopg2.connect(
    host=os.getenv("PGHOST", "").strip(),
    database=os.getenv("PGDATABASE", "").strip(),
    user=os.getenv("PGUSER", "").strip(),
    password=os.getenv("PGPASSWORD", "").strip(),
    port=int(os.getenv("PGPORT", "5432").strip()),
    sslmode=os.getenv("PGSSLMODE", "require").strip() or "require",
)
conn.autocommit = False
insert_sql = """
INSERT INTO tickets (
    ticket_number,
    ticket_year,
    ticket_sequence,
    direction,
    created_at,
    job_id,
    job_code_snapshot,
    job_name_snapshot,
    tax_exempt,
    customer_snapshot,
    truck_id,
    truck_number_snapshot,
    material_id,
    material_name_snapshot,
    quantity,
    unit,
    cost,
    notes,
    pdf_path,
    pdf_blob
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
ON CONFLICT (ticket_number) DO NOTHING
"""
inserted = 0
skipped = 0
errors = []
try:
    with conn.cursor() as cursor:
        trucks_map, customers_map, materials_map, jobs = build_lookup_maps(cursor)
        for idx, row in df_src.iterrows():
            try:
                created_dt = parse_created_at(row)
                direction = normalize_direction(row.get(COL_MAP.get("direction"), "IN"))
                raw_ticket_number = row.get(COL_MAP.get("ticket_number"), "")
                ticket_number, ticket_year, ticket_seq = allocate_ticket_number(cursor, created_dt, raw_ticket_number)
                job_code_in = clean_text(row.get(COL_MAP.get("job_code"), ""))
                job_name_in = clean_text(row.get(COL_MAP.get("job_name"), ""))
                customer_in = clean_text(row.get(COL_MAP.get("customer"), ""))
                tax_exempt_in = clean_text(row.get(COL_MAP.get("tax_exempt"), ""))
                job = find_job(jobs, job_code_in, job_name_in)
                job_id = job["id"] if job else None
                job_code_snapshot = (job["job_code"] if job and job["job_code"] else job_code_in) or "UNKNOWN"
                job_name_snapshot = (job["job_name"] if job and job["job_name"] else job_name_in) or "UNKNOWN"
                customer_snapshot = (customer_in or (job["customer"] if job else "")).strip()
                tax_exempt_snapshot = (tax_exempt_in or (job["tax_exempt"] if job else "") or "New").strip()
                truck_number_snapshot = clean_text(row.get(COL_MAP.get("truck_number"), ""))
                truck_id = trucks_map.get(clean_key(truck_number_snapshot))
                material_name_snapshot = clean_text(row.get(COL_MAP.get("material_name"), ""))
                material_id = materials_map.get((clean_key(material_name_snapshot), direction))
                if material_id is None:
                    material_id = materials_map.get((clean_key(material_name_snapshot), "*"))
                quantity = to_float(row.get(COL_MAP.get("quantity"), 1), 1.0)
                unit = clean_text(row.get(COL_MAP.get("unit"), "")) or "Load"
                cost = to_float(row.get(COL_MAP.get("cost"), 0), 0.0)
                notes = clean_text(row.get(COL_MAP.get("notes"), "")) or ""
                values = (
                    ticket_number,
                    ticket_year,
                    ticket_seq,
                    direction,
                    created_dt.isoformat(timespec="seconds"),
                    job_id,
                    job_code_snapshot,
                    job_name_snapshot,
                    tax_exempt_snapshot,
                    customer_snapshot,
                    truck_id,
                    truck_number_snapshot,
                    material_id,
                    material_name_snapshot,
                    quantity,
                    unit,
                    cost,
                    notes,
                    None,   # pdf_path (legacy rows can be NULL)
                    None,   # pdf_blob (legacy rows can be NULL)
                )
                if DRY_RUN:
                    inserted += 1
                else:
                    cursor.execute(insert_sql, values)
                    if cursor.rowcount == 1:
                        inserted += 1
                    else:
                        skipped += 1
            except Exception as row_exc:
                errors.append((int(idx), str(row_exc)))
        if DRY_RUN:
            conn.rollback()
        else:
            conn.commit()
    print(f"DRY_RUN={DRY_RUN}")
    print(f"Rows processed: {len(df_src)}")
    print(f"Rows ready/inserted: {inserted}")
    print(f"Rows skipped (duplicate ticket_number): {skipped}")
    print(f"Rows with errors: {len(errors)}")
    if errors:
        print("First 10 errors:")
        for i, msg in errors[:10]:
            print(f"  - Row {i}: {msg}")
finally:
    conn.close()


Loaded rows: 2624
DRY_RUN=False
Rows processed: 2624
Rows ready/inserted: 2624
Rows skipped (duplicate ticket_number): 0
Rows with errors: 0


In [ ]:
# Add a new column in material_price and manually set values

conn = psycopg2.connect(
    host=os.getenv("PGHOST", "").strip(),
    database=os.getenv("PGDATABASE", "").strip(),
    user=os.getenv("PGUSER", "").strip(),
    password=os.getenv("PGPASSWORD", "").strip(),
    port=int(os.getenv("PGPORT", "5432").strip()),
    sslmode=os.getenv("PGSSLMODE", "require").strip() or "require",
)

cur = conn.cursor()

# 1) Add new column
cur.execute("""
ALTER TABLE material_price
ADD COLUMN IF NOT EXISTS price_category TEXT;
""")

# 2) Manually enter values (edit this list as needed)
manual_values = [
    ("dirt", "IN", "standard"),
    ("dirt", "OUT", "premium"),
    ("rock", "IN", "premium"),
]

# 3) Update rows
for material_name, direction, category in manual_values:
    cur.execute(
        """
        UPDATE material_price
        SET price_category = %s
        WHERE LOWER(material) = LOWER(%s)
          AND UPPER(direction) = UPPER(%s);
        """,
        (category, material_name, direction),
    )

conn.commit()

# 4) Verify
cur.execute("""
SELECT id, material, direction, price_category
FROM material_price
ORDER BY id
LIMIT 50;
""")
for row in cur.fetchall():
    print(row)

cur.close()
conn.close()